In [ ]:
from thesis_utils import APP_ROOT, DATA_ROOT, PREVIEW_ROWS, PROJECT_ROOT, paths
from pathlib import Path
import pandas as pd

currentDir = APP_ROOT

DATA_DIR = paths.TRAINING_DATASETS_DIR 

TARGET_PATH = paths.TARGET_LEVEL_SENTIMENT_DATASET_PARQUET_PATH
TARGET_CSV_PATH = paths.TARGET_LEVEL_SENTIMENT_DATASET_CSV_PATH

if TARGET_PATH.exists():
    target_level_sentiment_df = pd.read_parquet(TARGET_PATH)
    print("Parquet okundu:", TARGET_PATH)
elif TARGET_CSV_PATH.exists():
    target_level_sentiment_df = pd.read_csv(TARGET_CSV_PATH)
    print("CSV okundu:", TARGET_CSV_PATH)
else:
    raise FileNotFoundError(
        f"target_level_sentiment_df bulunamadı.\nAranan yollar:\n{TARGET_PATH}\n{TARGET_CSV_PATH}"
    )

print("Shape:", target_level_sentiment_df.shape)

print("\nColumns:")
print(target_level_sentiment_df.columns.tolist())

print("\nHead:")


In [ ]:
target_level_sentiment_df.dataset_full_name.unique()

In [ ]:
target_level_sentiment_df.tail(1)

In [ ]:
# ============================================================
# SPLIT CELL
# target_level_sentiment_df -> train_df / val_df / test_df
#
# Model hücresi için gerekli kolonlar:
# input_text
# label
# ============================================================

import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

if "target_level_sentiment_df" not in globals():
    raise ValueError("target_level_sentiment_df bulunamadı. Önce dataframe'i oku.")

df = target_level_sentiment_df.copy()

# ------------------------------------------------------------
# 1) Geçerli text / label filtrele
# ------------------------------------------------------------
valid_labels = ["negative", "neutral", "positive"]

df["text"] = df["text"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.lower().str.strip()

df = df[
    (df["text"] != "") &
    (df["label"].isin(valid_labels))
].copy()

print("Filtered shape:", df.shape)
print("\nLabel distribution:")
print(df["label"].value_counts())

# ------------------------------------------------------------
# 2) input_text oluştur
# Target-level veri olduğu için target bilgisini input'a ekliyoruz
# ------------------------------------------------------------
for col in ["target", "entity", "aspect"]:
    if col not in df.columns:
        df[col] = ""

df["target_for_input"] = (
    df["target"]
    .fillna(df["entity"])
    .fillna("")
    .astype(str)
    .str.strip()
)

df["aspect_for_input"] = (
    df["aspect"]
    .fillna("")
    .astype(str)
    .str.strip()
)

def build_input_text(row):
    text = str(row["text"]).strip()
    target = str(row["target_for_input"]).strip()
    aspect = str(row["aspect_for_input"]).strip()

    parts = [text]

    if target and target.lower() not in ["none", "nan"]:
        parts.append(f"Target: {target}")

    if aspect and aspect.lower() not in ["none", "nan"]:
        parts.append(f"Aspect: {aspect}")

    return " [SEP] ".join(parts)

df["input_text"] = df.apply(build_input_text, axis=1)

# ------------------------------------------------------------
# 3) Aynı text train/test'e dağılmasın diye text bazlı group split
# ------------------------------------------------------------
df["text_group"] = df["text"].astype(str).str.lower().str.strip()

group_df = (
    df.groupby("text_group")["label"]
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
    .rename(columns={"label": "group_label"})
)

print("\nUnique text groups:", len(group_df))
print("\nGroup label distribution:")
print(group_df["group_label"].value_counts())

# ------------------------------------------------------------
# 4) 70 / 15 / 15 split
# ------------------------------------------------------------
train_groups, temp_groups = train_test_split(
    group_df,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=group_df["group_label"]
)

val_groups, test_groups = train_test_split(
    temp_groups,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_groups["group_label"]
)

train_group_set = set(train_groups["text_group"])
val_group_set = set(val_groups["text_group"])
test_group_set = set(test_groups["text_group"])

train_df = df[df["text_group"].isin(train_group_set)].copy()
val_df = df[df["text_group"].isin(val_group_set)].copy()
test_df = df[df["text_group"].isin(test_group_set)].copy()

# ------------------------------------------------------------
# 5) Yardımcı kolonları temizle
# ------------------------------------------------------------
drop_cols = ["target_for_input", "aspect_for_input", "text_group"]

train_df = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns]).reset_index(drop=True)
val_df = val_df.drop(columns=[c for c in drop_cols if c in val_df.columns]).reset_index(drop=True)
test_df = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns]).reset_index(drop=True)

# ------------------------------------------------------------
# 6) Kontroller
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("SPLITS READY")
print("=" * 80)

print("train_df:", train_df.shape)
print("val_df  :", val_df.shape)
print("test_df :", test_df.shape)

print("\nTrain label:")
print(train_df["label"].value_counts())

print("\nVal label:")
print(val_df["label"].value_counts())

print("\nTest label:")
print(test_df["label"].value_counts())

# Leakage kontrolü
train_texts = set(train_df["text"].astype(str).str.lower().str.strip())
val_texts = set(val_df["text"].astype(str).str.lower().str.strip())
test_texts = set(test_df["text"].astype(str).str.lower().str.strip())

print("\nLeakage check:")
print("train-val overlap :", len(train_texts.intersection(val_texts)))
print("train-test overlap:", len(train_texts.intersection(test_texts)))
print("val-test overlap  :", len(val_texts.intersection(test_texts)))

print("\nTrain sample:")
display(train_df[["input_text", "label"]].head(PREVIEW_ROWS))

print("\nVal sample:")
display(val_df[["input_text", "label"]].head(PREVIEW_ROWS))

print("\nTest sample:")
display(test_df[["input_text", "label"]].head(PREVIEW_ROWS))

In [ ]:
# ============================================================
# FINBERT TARGET-LEVEL SENTIMENT FINE-TUNE
# CHECKPOINT AÇIK - KALDIĞI YERDEN DEVAM EDER
#
# Gerekli hazır değişkenler:
# train_df, val_df, test_df
#
# Gerekli kolonlar:
# input_text
# label  -> negative / neutral / positive
# ============================================================

import os
import gc
import inspect
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch import nn
from datasets import Dataset

from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

from transformers.trainer_utils import get_last_checkpoint

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# ============================================================
# 0) AYARLAR
# ============================================================

MODEL_NAME = "ProsusAI/finbert"

TEXT_INPUT_COL = "input_text"
STRING_LABEL_COL = "label"

MAX_LENGTH = 128
RANDOM_STATE = 42

EPOCHS = 4
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_STATE)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ============================================================
# 0.1) KALICI CHECKPOINT KLASÖRLERİ
# ============================================================

currentDir = APP_ROOT

RUN_DIR = currentDir / "trained_models" / "finbert_target_level_sentiment"
BASE_EVAL_DIR = RUN_DIR / "base_eval"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
FINAL_MODEL_DIR = RUN_DIR / "final_model"

BASE_EVAL_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("\nKalıcı klasörler:")
print("RUN_DIR        :", RUN_DIR)
print("BASE_EVAL_DIR  :", BASE_EVAL_DIR)
print("CHECKPOINT_DIR :", CHECKPOINT_DIR)
print("FINAL_MODEL_DIR:", FINAL_MODEL_DIR)

# ============================================================
# 1) GÜVENLİK KONTROLÜ
# ============================================================

for df_name in ["train_df", "val_df", "test_df"]:
    if df_name not in globals():
        raise ValueError(f"{df_name} bulunamadı. Önce split hücresini çalıştır.")

for name, split_df in {
    "train_df": train_df,
    "val_df": val_df,
    "test_df": test_df,
}.items():
    print(name, split_df.shape)

    for col in [TEXT_INPUT_COL, STRING_LABEL_COL]:
        if col not in split_df.columns:
            raise ValueError(f"{name} içinde eksik kolon: {col}")

print("\nTrain sample:")
display(train_df[[TEXT_INPUT_COL, STRING_LABEL_COL]].head(PREVIEW_ROWS))

# ============================================================
# 2) FINBERT LABEL SIRASINI OKU
# ============================================================

config = AutoConfig.from_pretrained(MODEL_NAME)

print("\nOriginal FinBERT config id2label:")
print(config.id2label)


def normalize_label_name(x):
    s = str(x).strip().lower()

    if "negative" in s or s == "neg":
        return "negative"
    if "neutral" in s or s == "neu":
        return "neutral"
    if "positive" in s or s == "pos":
        return "positive"

    return None


finbert_id2label = {}

for k, v in config.id2label.items():
    norm = normalize_label_name(v)
    if norm is not None:
        finbert_id2label[int(k)] = norm

if set(finbert_id2label.values()) != {"negative", "neutral", "positive"}:
    print("\nUYARI: FinBERT config label isimleri net okunamadı.")
    print("Standart mapping kullanılacak: 0 negative, 1 neutral, 2 positive")

    finbert_id2label = {
        0: "negative",
        1: "neutral",
        2: "positive",
    }

finbert_label2id = {v: k for k, v in finbert_id2label.items()}

print("\nKullanılacak label mapping:")
print("id2label:", finbert_id2label)
print("label2id:", finbert_label2id)

NUM_LABELS = 3

# ============================================================
# 3) TRAINER INIT UYUMLULUK FONKSİYONU
# ============================================================

def get_trainer_tokenizer_kwargs(tokenizer):
    sig = inspect.signature(Trainer.__init__)

    if "processing_class" in sig.parameters:
        return {"processing_class": tokenizer}

    if "tokenizer" in sig.parameters:
        return {"tokenizer": tokenizer}

    return {}

# ============================================================
# 4) DATASET HAZIRLAMA
# ============================================================

def prepare_split_for_finbert(split_df, split_name):
    temp = split_df[[TEXT_INPUT_COL, STRING_LABEL_COL]].copy()

    temp[TEXT_INPUT_COL] = temp[TEXT_INPUT_COL].astype(str).str.strip()
    temp[STRING_LABEL_COL] = temp[STRING_LABEL_COL].astype(str).str.strip().str.lower()

    temp = temp[temp[TEXT_INPUT_COL] != ""].copy()
    temp = temp[temp[STRING_LABEL_COL].isin(["negative", "neutral", "positive"])].copy()

    temp["labels"] = temp[STRING_LABEL_COL].map(finbert_label2id).astype(int)

    print(f"\n{split_name} prepared shape:", temp.shape)
    print(temp[STRING_LABEL_COL].value_counts())

    return temp.reset_index(drop=True)


train_data = prepare_split_for_finbert(train_df, "train")
val_data = prepare_split_for_finbert(val_df, "val")
test_data = prepare_split_for_finbert(test_df, "test")

print("\nNumeric label distribution:")
display(pd.DataFrame({
    "train": train_data["labels"].value_counts().sort_index(),
    "val": val_data["labels"].value_counts().sort_index(),
    "test": test_data["labels"].value_counts().sort_index(),
}).fillna(0).astype(int))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def make_hf_dataset(dataframe):
    ds = Dataset.from_pandas(
        dataframe[[TEXT_INPUT_COL, "labels"]].reset_index(drop=True)
    )

    def tokenize_fn(batch):
        return tokenizer(
            batch[TEXT_INPUT_COL],
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
        )

    ds = ds.map(tokenize_fn, batched=True)

    keep_cols = ["input_ids", "attention_mask", "labels"]

    if "token_type_ids" in ds.column_names:
        keep_cols.append("token_type_ids")

    ds.set_format(type="torch", columns=keep_cols)

    return ds


train_ds = make_hf_dataset(train_data)
val_ds = make_hf_dataset(val_data)
test_ds = make_hf_dataset(test_data)

print("\nHF datasets hazır:")
print("Train:", train_ds)
print("Val  :", val_ds)
print("Test :", test_ds)

# ============================================================
# 5) METRIC / REPORT FONKSİYONLARI
# ============================================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }


def show_prediction_report(trainer, dataset, title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    pred_output = trainer.predict(dataset)

    logits = pred_output.predictions
    y_true = pred_output.label_ids
    y_pred = np.argmax(logits, axis=-1)

    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")

    print(f"Accuracy   : {acc:.4f}")
    print(f"F1 Macro   : {f1_macro:.4f}")
    print(f"F1 Weighted: {f1_weighted:.4f}")

    label_ids_sorted = sorted(finbert_id2label.keys())
    target_names = [finbert_id2label[i] for i in label_ids_sorted]

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=label_ids_sorted,
            target_names=target_names,
            digits=4,
            zero_division=0,
        )
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=label_ids_sorted,
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{finbert_id2label[i]}" for i in label_ids_sorted],
        columns=[f"pred_{finbert_id2label[i]}" for i in label_ids_sorted],
    )

    print("\nConfusion Matrix:")
    display(cm_df)

    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "y_true": y_true,
        "y_pred": y_pred,
        "confusion_matrix": cm_df,
    }

# ============================================================
# 6) TRAINING ARGUMENTS - CHECKPOINT AÇIK
# ============================================================

def build_training_args(output_dir, train=True):
    sig = inspect.signature(TrainingArguments.__init__)

    kwargs = dict(
        output_dir=str(output_dir),
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        report_to="none",
        seed=RANDOM_STATE,
    )

    if train:
        kwargs.update(
            learning_rate=LEARNING_RATE,
            per_device_train_batch_size=TRAIN_BATCH_SIZE,
            num_train_epochs=EPOCHS,
            weight_decay=WEIGHT_DECAY,
            logging_steps=50,
        )

        # Eval ve save aynı strateji olmalı.
        if "eval_strategy" in sig.parameters:
            kwargs["eval_strategy"] = "epoch"
        elif "evaluation_strategy" in sig.parameters:
            kwargs["evaluation_strategy"] = "epoch"

        if "save_strategy" in sig.parameters:
            kwargs["save_strategy"] = "epoch"

        if "save_total_limit" in sig.parameters:
            kwargs["save_total_limit"] = 3

        if "load_best_model_at_end" in sig.parameters:
            kwargs["load_best_model_at_end"] = True

        if "metric_for_best_model" in sig.parameters:
            kwargs["metric_for_best_model"] = "f1_macro"

        if "greater_is_better" in sig.parameters:
            kwargs["greater_is_better"] = True

        if "logging_strategy" in sig.parameters:
            kwargs["logging_strategy"] = "steps"

    else:
        kwargs.update(
            per_device_train_batch_size=TRAIN_BATCH_SIZE,
            num_train_epochs=1,
        )

        if "eval_strategy" in sig.parameters:
            kwargs["eval_strategy"] = "no"
        elif "evaluation_strategy" in sig.parameters:
            kwargs["evaluation_strategy"] = "no"

        if "save_strategy" in sig.parameters:
            kwargs["save_strategy"] = "no"

        if "logging_strategy" in sig.parameters:
            kwargs["logging_strategy"] = "no"

    if "fp16" in sig.parameters:
        kwargs["fp16"] = torch.cuda.is_available()

    return TrainingArguments(**kwargs)

# ============================================================
# 7) BASELINE: FINE-TUNE EDİLMEMİŞ FINBERT TEST
# ============================================================

print("\n" + "=" * 100)
print("BASELINE: Fine-tune edilmemiş FinBERT test ediliyor")
print("=" * 100)

base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
base_model.to(device)

base_args = build_training_args(BASE_EVAL_DIR, train=False)

base_trainer = Trainer(
    model=base_model,
    args=base_args,
    compute_metrics=compute_metrics,
    **get_trainer_tokenizer_kwargs(tokenizer),
)

baseline_result = show_prediction_report(
    base_trainer,
    test_ds,
    "BASELINE TEST RESULT - ORIGINAL FINBERT"
)

del base_model
del base_trainer
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ============================================================
# 8) CLASS WEIGHT HAZIRLA
# ============================================================

train_labels = train_data["labels"].values
counts = np.bincount(train_labels, minlength=NUM_LABELS)

class_weights = []

for class_id in range(NUM_LABELS):
    if counts[class_id] == 0:
        class_weights.append(1.0)
    else:
        class_weights.append(len(train_labels) / (NUM_LABELS * counts[class_id]))

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

print("\nClass counts:", counts)
print("Class weights:")

for i, w in enumerate(class_weights):
    print(i, finbert_id2label[i], "->", round(float(w), 4))

# ============================================================
# 9) WEIGHTED TRAINER
# ============================================================

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")

        outputs = model(**inputs)
        logits = outputs.get("logits")

        weights = class_weights_tensor.to(logits.device)

        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

# ============================================================
# 10) FINE-TUNE
# ============================================================

print("\n" + "=" * 100)
print("FINE-TUNE BAŞLIYOR - CHECKPOINT AÇIK")
print("=" * 100)

fine_tune_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
)

fine_tune_model.config.id2label = finbert_id2label
fine_tune_model.config.label2id = finbert_label2id

fine_tune_model.to(device)

training_args = build_training_args(CHECKPOINT_DIR, train=True)

trainer = WeightedTrainer(
    model=fine_tune_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    **get_trainer_tokenizer_kwargs(tokenizer),
)

# ============================================================
# 10.1) VARSA CHECKPOINT'TEN DEVAM ET
# ============================================================

last_checkpoint = None

if CHECKPOINT_DIR.exists():
    last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR))

if last_checkpoint is not None:
    print("\nCheckpoint bulundu. Eğitim buradan devam edecek:")
    print(last_checkpoint)
    train_output = trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("\nCheckpoint bulunamadı. Eğitim sıfırdan başlayacak.")
    train_output = trainer.train()

print("\nTraining output:")
print(train_output)

# ============================================================
# 11) EN İYİ / SON MODELİ KAYDET
# ============================================================

print("\nFinal model kaydediliyor:")
print(FINAL_MODEL_DIR)

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

print("Final model kaydedildi.")

# ============================================================
# 12) VALIDATION VE TEST SONUCU
# ============================================================

val_result = show_prediction_report(
    trainer,
    val_ds,
    "FINE-TUNED FINBERT - VALIDATION RESULT"
)

test_result = show_prediction_report(
    trainer,
    test_ds,
    "FINE-TUNED FINBERT - TEST RESULT"
)

# ============================================================
# 13) BASELINE VS FINE-TUNED ÖZET
# ============================================================

summary_df = pd.DataFrame([
    {
        "model": "Original FinBERT",
        "test_accuracy": baseline_result["accuracy"],
        "test_f1_macro": baseline_result["f1_macro"],
        "test_f1_weighted": baseline_result["f1_weighted"],
    },
    {
        "model": "Fine-tuned FinBERT",
        "test_accuracy": test_result["accuracy"],
        "test_f1_macro": test_result["f1_macro"],
        "test_f1_weighted": test_result["f1_weighted"],
    }
])

summary_df["delta_accuracy"] = summary_df["test_accuracy"] - summary_df.loc[0, "test_accuracy"]
summary_df["delta_f1_macro"] = summary_df["test_f1_macro"] - summary_df.loc[0, "test_f1_macro"]
summary_df["delta_f1_weighted"] = summary_df["test_f1_weighted"] - summary_df.loc[0, "test_f1_weighted"]

print("\n" + "=" * 100)
print("BASELINE VS FINE-TUNED SUMMARY")
print("=" * 100)

display(summary_df)

# ============================================================
# 14) TAMAM
# ============================================================

print("\nTamamlandı.")
print("Checkpoint klasörü korundu:")
print(CHECKPOINT_DIR)
print("\nFinal model klasörü:")
print(FINAL_MODEL_DIR)